In [55]:
import os
from glob import glob
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
import cv2
from tqdm import tqdm
import shutil
import xml.etree.ElementTree as ET
import ultralytics
from ultralytics import YOLO
import yaml
import torch

## Import a pre-trained yolov8 model

In [4]:
# Import the yolov8 model

model = YOLO("yolov8n.pt")  # load a pretrained model (recommended for training)
model.info()
model.names

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

## Train the model on the dataset

In [38]:
TRAIN_PATH = "data_processed/train/images"
GROUND_TRUTH_PATH = "data_processed/train/images_annotated"

In [45]:
# Separate the training set into training (80%) and validation (20%) sets by SEQUENCE folders
train_base = "data_processed/train"
sequences = sorted([d for d in os.listdir(os.path.join(train_base, 'images')) 
                   if os.path.isdir(os.path.join(train_base, 'images', d))])

# Split sequences (not individual images)
np.random.shuffle(sequences)
split_idx = int(0.8 * len(sequences))
train_sequences = sequences[:split_idx]
val_sequences = sequences[split_idx:]

print(f"Total sequences: {len(sequences)}")
print(f"Train sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")

Total sequences: 60
Train sequences: 48
Validation sequences: 12


In [46]:
# Move validation sequence folders to val/images
val_images_path = "data_processed/val/images"
os.makedirs(val_images_path, exist_ok=True)

train_images_path = os.path.join(train_base, 'images')
train_labels_path = os.path.join(train_base, 'labels')

for seq in val_sequences:
    # Move images
    src_img = os.path.join(train_images_path, seq)
    dst_img = os.path.join(val_images_path, seq)
    if os.path.exists(src_img):
        shutil.move(src_img, dst_img)
    
    # Move labels
    src_label = os.path.join(train_labels_path, seq)
    dst_label = os.path.join("data_processed/val", 'labels', seq)
    if os.path.exists(src_label):
        os.makedirs(os.path.dirname(dst_label), exist_ok=True)
        shutil.move(src_label, dst_label)

print(f"Moved {len(val_sequences)} validation sequences")

Moved 12 validation sequences


In [47]:
# No additional extraction needed - validation images are already in val/images/
print("Validation images are ready in data_processed/val/images/")

Validation images are ready in data_processed/val/images/


In [48]:
# Create the yaml file for training
abs_path = os.path.abspath('data_processed')
data_yaml = {
    'path': abs_path,
    'train': os.path.join(abs_path, 'train', 'images'),
    'val': os.path.join(abs_path, 'val', 'images'),  # Use validation images
    'nc': 3,
    'names': ['car', 'bus', 'van']
}

yaml_path = os.path.join(abs_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)
print(f'Data YAML saved to {yaml_path}')

Data YAML saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\data.yaml


In [ ]:
# Check for GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

# Train the model with aggressive speed optimizations
model.train(
    data="data_processed/data.yaml",
    epochs=10,              # Further reduced from 25 (2.5x faster)
    imgsz=320,              # Further reduced from 416 (1.7x faster per image)
    batch=64,               # Increased from 16 (faster batches)
    device=device,          # Use GPU if available
    patience=5,             # Stop early after 5 epochs of no improvement
    amp=True,               # Automatic Mixed Precision (faster on GPU)
    name="yolov8n_vehicle_detection_ultra_fast"
)

# Evaluate the model
metrics = model.val()
print(metrics)

Training on: cpu
New https://pypi.org/project/ultralytics/8.4.2 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.235  Python-3.11.14 torch-2.9.1+cpu CPU (11th Gen Intel Core i7-11370H @ 3.30GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_processed/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_vehi